In [1]:
%pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-chroma langchain-google-genai chromadb pymupdf pydantic python-dotenv

   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ------------------ --------------------- 262.1/580.5 kB ? eta -:--:--
   ------------------ --------------------- 262.1/580.5 kB ? eta -:--:--
   ---------------------------------------- 580.5/580.5 kB 887.0 kB/s  0:00:01
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.3 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.3 MB 2.2 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.3 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 2.9 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogen-core 0.7.5 requires protobuf~=5.29.3, but you have protobuf 6.33.6 which is incompatible.
crewai 1.15.5 requires aiofiles~=24.1.0, but you have aiofiles 25.1.0 which is incompatible.
crewai 1.15.5 requires chromadb~=1.1.0, but you have chromadb 1.5.9 which is incompatible.
crewai 1.15.5 requires pydantic<2.13,>=2.11.9, but you have pydantic 2.14.3 which is incompatible.
crewai 1.15.5 requires tokenizers<1,>=0.21, but you have tokenizers 0.20.3 which is incompatible.
crewai-cli 1.15.5 requires pydantic<2.13,>=2.11.9, but you have pydantic 2.14.3 which is incompatible.
crewai-core 1.15.5 requires pydantic<2.13,>=2.11.9, but you have pydantic 2.14.3 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have

In [2]:
import os
import json
import fitz

from dotenv import load_dotenv

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_chroma import Chroma

from langchain_core.documents import Document

from pydantic import BaseModel, Field

C:\Python313\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [5]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

assert api_key, "Gemini API key not found!"

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key,
    temperature=0.2
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=api_key
)

print("Gemini LLM and Embeddings initialized!")

Gemini LLM and Embeddings initialized!


In [10]:
response = llm.invoke(
    "Explain what a resume-JD matching system does in 3 lines."
)

answer = "\n".join(
    block["text"]
    for block in response.content
    if isinstance(block, dict) and block.get("type") == "text"
)

print(answer)

It extracts key skills, qualifications, and experience from both resumes and job descriptions.
It uses algorithms or AI to compare a candidate's profile against the specific requirements of the role.
It generates a compatibility score and ranks applicants to help recruiters quickly identify the best-fit talent.


In [11]:
pdf_path = "SyedResume.pdf"

pdf = fitz.open(pdf_path)

resume_documents = []

for page_number, page in enumerate(pdf):

    text = page.get_text("text").strip()

    if text:

        document = Document(
            page_content=text,
            metadata={
                "page": page_number + 1,
                "source": pdf_path,
                "section": "resume"
            }
        )

        resume_documents.append(document)

pdf.close()

resume_text = "\n\n".join(
    doc.page_content
    for doc in resume_documents
)

print(resume_text)

SYED ABDUL KATHIR A 
                          syedabdul122001@gmail.com | LinkedIn | GitHub | 9003550500 
CAREER OBJECTIVE:  
Entry-level Software Developer with strong proficiency in Python, Django, and SQL, and a solid understanding 
of web technologies. Seeking an opportunity to contribute to scalable application development, enhance 
problem-solving skills, and grow as a software engineer while delivering reliable and maintainable solutions. 
 
                                                                                              EDUCATION 
• Syed ammal Engineering College                                                                                                      Ramanathapuram,India                      
Bachelour of Technology in Artificial Intelligence and Data Science; CGPA 7.9                 Nov 2021 – May 2025 
• Velammal Matric HR Sec School                                                                                                       Thirupuvanam,In

In [12]:
print("Total characters:", len(resume_text))

print("Total pages:", len(resume_documents))

print("\nFirst 1000 characters:\n")

print(resume_text[:1000])

Total characters: 5700
Total pages: 2

First 1000 characters:

SYED ABDUL KATHIR A 
                          syedabdul122001@gmail.com | LinkedIn | GitHub | 9003550500 
CAREER OBJECTIVE:  
Entry-level Software Developer with strong proficiency in Python, Django, and SQL, and a solid understanding 
of web technologies. Seeking an opportunity to contribute to scalable application development, enhance 
problem-solving skills, and grow as a software engineer while delivering reliable and maintainable solutions. 
 
                                                                                              EDUCATION 
• Syed ammal Engineering College                                                                                                      Ramanathapuram,India                      
Bachelour of Technology in Artificial Intelligence and Data Science; CGPA 7.9                 Nov 2021 – May 2025 
• Velammal Matric HR Sec School                                                       

In [13]:
class Project(BaseModel):

    name: str = Field(
        description="Project name"
    )

    technologies: list[str] = Field(
        description="Technologies used"
    )

    description: str = Field(
        description="Project description"
    )


class ResumeInformation(BaseModel):

    candidate_skills: list[str]

    education: list[str]

    certifications: list[str]

    projects: list[Project]

In [14]:
structured_llm = llm.with_structured_output(
    ResumeInformation
)

resume_extraction_prompt = f"""
You are a resume information extraction system.

Extract the following from the resume:

1. Technical skills
2. Education
3. Certifications
4. Projects
5. Technologies used in each project
6. Project descriptions

Important:
- Extract only information explicitly present.
- Do not invent skills.
- Do not assume technologies.
- Keep project descriptions faithful to the resume.

RESUME:

{resume_text}
"""

resume_info = structured_llm.invoke(
    resume_extraction_prompt
)

resume_data = resume_info.model_dump()

print(
    json.dumps(
        resume_data,
        indent=4
    )
)

{
    "candidate_skills": [
        "Python",
        "Django (Models,Views,Templates,ORM)",
        "HTML",
        "CSS",
        "Javascript",
        "ReactJS(Basics)",
        "SQL (Oracle SQL ,SQLite-Django ORM)",
        "Machine Learning (Basics)",
        "Cloud Fundementals (AWS)"
    ],
    "education": [
        "Bachelour of Technology in Artificial Intelligence and Data Science, Syed ammal Engineering College, CGPA 7.9 (Nov 2021 \u2013 May 2025)",
        "Higher Secondary, Velammal Matric HR Sec School, Percentage 83 % (June 2019 \u2013 May 2021)",
        "SSLC, Velammal Matric HR Sec School, Percentage 80 % (June 2018 \u2013 May 2019)"
    ],
    "certifications": [
        "Joy of Computing using Python (NPTEL)",
        "PostgreSQL (Infosys Springboard)",
        "Introduction to Data Analytics (IBM & Coursera)"
    ],
    "projects": [
        {
            "name": "Customer Segmentation using Machine Learning",
            "technologies": [
                "Machine

In [15]:
jd_text = """
Job Role: Junior Machine Learning Engineer

Required Skills:

- Python
- SQL
- Machine Learning
- Scikit-learn
- TensorFlow
- NLP
- FastAPI
- Model Deployment

Responsibilities:

- Develop machine learning models.
- Perform data preprocessing.
- Build REST APIs for model deployment.
- Work with structured datasets.
- Evaluate model performance.

Preferred:
- Experience with Generative AI.
- Knowledge of RAG applications.
"""

In [16]:
class JobRequirements(BaseModel):

    job_role: str

    required_skills: list[str]

    preferred_skills: list[str]

    responsibilities: list[str]

    qualifications: list[str]


structured_jd_llm = llm.with_structured_output(
    JobRequirements
)

jd_requirements = structured_jd_llm.invoke(
    f"""
    Extract job requirements into structured fields.

    Separate mandatory skills from preferred skills.

    JOB DESCRIPTION:
    {jd_text}
    """
)

jd_data = jd_requirements.model_dump()

print(
    json.dumps(
        jd_data,
        indent=4
    )
)

{
    "job_role": "Junior Machine Learning Engineer",
    "required_skills": [
        "Python",
        "SQL",
        "Machine Learning",
        "Scikit-learn",
        "TensorFlow",
        "NLP",
        "FastAPI",
        "Model Deployment"
    ],
    "preferred_skills": [
        "Experience with Generative AI",
        "Knowledge of RAG applications"
    ],
    "responsibilities": [
        "Develop machine learning models.",
        "Perform data preprocessing.",
        "Build REST APIs for model deployment.",
        "Work with structured datasets.",
        "Evaluate model performance."
    ],
    "qualifications": []
}


In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

resume_chunks = text_splitter.split_documents(
    resume_documents
)

print("Total chunks:", len(resume_chunks))

Total chunks: 14


In [18]:
for index, chunk in enumerate(resume_chunks):

    print(f"\nCHUNK {index + 1}")

    print(chunk.page_content)

    print("Metadata:", chunk.metadata)

    print("-" * 60)


CHUNK 1
SYED ABDUL KATHIR A 
                          syedabdul122001@gmail.com | LinkedIn | GitHub | 9003550500 
CAREER OBJECTIVE:  
Entry-level Software Developer with strong proficiency in Python, Django, and SQL, and a solid understanding 
of web technologies. Seeking an opportunity to contribute to scalable application development, enhance 
problem-solving skills, and grow as a software engineer while delivering reliable and maintainable solutions.
Metadata: {'page': 1, 'source': 'SyedResume.pdf', 'section': 'resume'}
------------------------------------------------------------

CHUNK 2
EDUCATION 
• Syed ammal Engineering College                                                                                                      Ramanathapuram,India                      
Bachelour of Technology in Artificial Intelligence and Data Science; CGPA 7.9                 Nov 2021 – May 2025
Metadata: {'page': 1, 'source': 'SyedResume.pdf', 'section': 'resume'}
--------------------------

In [19]:
sample_text = [
    "Python is used for machine learning development.",
    "FastAPI is used to build REST APIs.",
    "SQL is used for database queries."
]

vectors = embeddings.embed_documents(
    sample_text
)

print("Number of vectors:", len(vectors))

print("Embedding dimensions:", len(vectors[0]))

print("First vector preview:", vectors[0][:10])

Number of vectors: 3
Embedding dimensions: 3072
First vector preview: [-0.008468057, 0.012395825, -0.01557523, -0.07063846, -0.010269813, 0.0074854507, -0.0016787818, 0.014564955, 0.0016024276, -0.008646691]


In [20]:
import uuid

collection_name = "resume_" + uuid.uuid4().hex[:12]

vectorstore = Chroma.from_documents(
    documents=resume_chunks,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory="./resume_chroma"
)

print("Resume successfully stored in ChromaDB!")

Resume successfully stored in ChromaDB!


In [21]:
query = "Which projects demonstrate machine learning?"

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

retrieved_docs = retriever.invoke(query)

for index, doc in enumerate(retrieved_docs):

    print(f"\nRetrieved result {index + 1}")

    print(doc.page_content)

    print("Page:", doc.metadata.get("page"))


Retrieved result 1
PROJECTS 
Customer Segmentation using Machine Learning                                                  October 2023 
• Developed a machine learning model using K-Means clustering to segment mall customers based on annual 
income, spending score, and demographic attributes. 
• Conducted extensive data preprocessing, including handling missing values, encoding categorical variables, 
and feature scaling, improving clustering accuracy and interpretability.
Page: 2

Retrieved result 2
by 15% and improve customer retention. 
AI Doorlock System using Face Detection                                                                                                May 2024 
•Developed a face detection-based door lock system by integrating Python (OpenCV) for real-time facial 
recognition with Arduino for hardware control. 
•Programmed the locking mechanism to grant access only to authorized individuals, ensuring quick and secure 
authentication.
Page: 2

Retrieved result 3
and

In [22]:
resume_skills = {
    skill.lower().strip()
    for skill in resume_data["candidate_skills"]
}

required_skills = {
    skill.lower().strip()
    for skill in jd_data["required_skills"]
}

preferred_skills = {
    skill.lower().strip()
    for skill in jd_data["preferred_skills"]
}

matched_skills = resume_skills & required_skills

missing_skills = required_skills - resume_skills

preferred_matches = resume_skills & preferred_skills

print("MATCHED:", matched_skills)

print("MISSING:", missing_skills)

print("PREFERRED MATCHES:", preferred_matches)

MATCHED: {'python'}
MISSING: {'model deployment', 'sql', 'fastapi', 'scikit-learn', 'nlp', 'machine learning', 'tensorflow'}
PREFERRED MATCHES: set()


In [23]:
if required_skills:

    compatibility = (
        len(matched_skills)
        / len(required_skills)
    ) * 100

else:

    compatibility = 0

print(
    f"Required skill coverage: {compatibility:.2f}%"
)

Required skill coverage: 12.50%


In [25]:
project_analysis_prompt = f"""
You are a technical recruiter.

Evaluate the candidate's projects against the JD.

For every project:

1. Explain its relevance.
2. Identify matching technical requirements.
3. Identify missing evidence.
4. Recommend improvements.

Do not invent project features or technologies.

RESUME PROJECTS:

{json.dumps(
    [p.model_dump() for p in resume_info.projects],
    indent=2
)}

JOB DESCRIPTION:

{jd_text}

Return a clear structured analysis.
"""

project_response = llm.invoke(
    project_analysis_prompt
)

answer = "\n".join(
    block["text"]
    for block in project_response.content
    if isinstance(block, dict) and block.get("type") == "text"
)

print(answer)

Here is the technical recruiter evaluation of the candidate's projects mapped against the Junior Machine Learning Engineer job description.

---

### **Project 1: Customer Segmentation using Machine Learning**

#### 1. Relevance
This project is highly relevant to the foundational responsibilities of the role. It showcases hands-on experience handling structured tabular datasets, performing end-to-end data preprocessing, developing an unsupervised machine learning model (K-Means), and translating algorithmic outputs into business impact.

#### 2. Matching Technical Requirements
* **Skills:** 
  * Machine Learning (K-Means Clustering)
* **Responsibilities:** 
  * **Develop machine learning models:** Built an unsupervised clustering model.
  * **Perform data preprocessing:** Handled missing values, encoded categorical variables, and executed feature scaling.
  * **Work with structured datasets:** Processed tabular customer demographic and behavioral data (income, spending score).

#### 3.

In [32]:
def retrieve_context(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    return context, docs

In [33]:
def resume_chat(question, chat_history):

    context, sources = retrieve_context(question)

    history = "\n".join(
        f"{message['role']}: {message['content']}"
        for message in chat_history
    )

    prompt = f"""
You are an expert Resume AI assistant.

Answer the user's question based on the
retrieved resume context and the job description.

Rules:

- Never invent candidate experience.
- Explain your reasoning.
- Identify actual evidence.
- Give practical recommendations.
- Admit when information is unavailable.

RESUME CONTEXT:

{context}

JOB DESCRIPTION:

{jd_text}

CHAT HISTORY:

{history}

QUESTION:

{question}
"""

    response = llm.invoke(prompt)

    return response.content, sources

In [34]:
chat_history = []

question = (
    "How can I improve my projects "
    "for this Machine Learning Engineer role?"
)

answer, sources = resume_chat(
    question,
    chat_history
)

print("AI RESPONSE:\n")

print(answer)

chat_history.append({
    "role": "user",
    "content": question
})

chat_history.append({
    "role": "assistant",
    "content": answer
})

AI RESPONSE:

[{'type': 'text', 'text': 'Based on a comparison between your resume context and the Junior Machine Learning Engineer job description, here is an analysis of your current projects along with actionable recommendations to improve them.\n\n---\n\n### 1. Analysis of Current Projects vs. Job Requirements\n\n#### **Project 1: Customer Segmentation using Machine Learning**\n* **Evidence in Resume:** Mentions K-Means clustering, structured customer data (income, spending score), and preprocessing (missing values, scaling, categorical encoding). There is also a fragmented bullet: *"by 15% and improve customer retention."*\n* **Reasoning:** \n  * **Strengths:** Demonstrates foundational machine learning, preprocessing, and working with structured data, which directly aligns with the job responsibilities.\n  * **Gaps:** It does not mention the specific libraries used (e.g., **Scikit-learn**), does not describe model evaluation metrics (e.g., Silhouette score), lacks **SQL** integra